# Baseline age-stratified survival analysis

This notebook compares recorded survival between smokers and non-smokers
within the four agreed age bands. Run it from the repository root in the
Pixi `smoke` environment. It follows
[the shared data preparation](smoking_is_bad.ipynb) and excludes records
with missing age, smoking status, or outcome. Salary is not used.

In [ ]:
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
from matplotlib.ticker import PercentFormatter

raw = pd.read_csv(Path("data") / "smoking_data.csv")
data = raw.drop(columns="Unnamed: 0", errors="ignore").copy()
data["smokes"] = data["smoker"].map({"No": 0, "Yes": 1})
data["alive"] = data["outcome"].map({"Dead": 0, "Alive": 1})

for original, encoded in [("smoker", "smokes"), ("outcome", "alive")]:
    invalid = data[original].notna() & data[encoded].isna()
    if invalid.any():
        raise ValueError(f"Unexpected categories in {original}: {data.loc[invalid, original].unique()}")

analysis = data.dropna(subset=["age", "smokes", "alive"]).copy()
print(f"Rows in source: {len(raw):,}")
print(f"Rows excluded for missing age, smoking status, or outcome: {len(raw) - len(analysis):,}")
print(f"Complete analysis rows: {len(analysis):,}")

In [ ]:
age_edges = [18, 30, 50, 65, float("inf")]
age_labels = ["18–29", "30–49", "50–64", "65+"]
analysis["age_band"] = pd.cut(
    analysis["age"], bins=age_edges, labels=age_labels, right=False
)
if analysis["age_band"].isna().any():
    raise ValueError("Age bands do not cover every complete-case observation.")

counts = (
    analysis.groupby(["age_band", "smokes"], observed=False)
    .agg(n=("alive", "size"), alive=("alive", "sum"))
)
wide_n = counts["n"].unstack("smokes").reindex(index=age_labels, columns=[0, 1])
wide_alive = counts["alive"].unstack("smokes").reindex(index=age_labels, columns=[0, 1])
if wide_n.isna().any().any() or (wide_n == 0).any().any():
    raise ValueError("Each age band must contain observations in both smoking groups.")

baseline_summary = pd.DataFrame(
    {
        "n_non_smokers": wide_n[0].astype(int),
        "alive_non_smokers": wide_alive[0].astype(int),
        "n_smokers": wide_n[1].astype(int),
        "alive_smokers": wide_alive[1].astype(int),
    },
    index=pd.Index(age_labels, name="age_band"),
)
baseline_summary["rate_non_smokers"] = baseline_summary["alive_non_smokers"] / baseline_summary["n_non_smokers"]
baseline_summary["rate_smokers"] = baseline_summary["alive_smokers"] / baseline_summary["n_smokers"]
baseline_summary["difference_pp"] = 100 * (
    baseline_summary["rate_smokers"] - baseline_summary["rate_non_smokers"]
)
display(baseline_summary.style.format({
    "rate_non_smokers": "{:.1%}",
    "rate_smokers": "{:.1%}",
    "difference_pp": "{:+.2f}",
}))

In [ ]:
overall = analysis.groupby("smokes")["alive"].agg(n="size", alive="sum", rate="mean")
age_65_plus = analysis["age_band"].eq("65+")
overall["age_65_plus_share"] = age_65_plus.groupby(analysis["smokes"]).mean()
crude_difference_pp = 100 * (overall.loc[1, "rate"] - overall.loc[0, "rate"])
print("Overall complete-case comparison (0 = non-smoker, 1 = smoker):")
display(overall.style.format({"rate": "{:.1%}", "age_65_plus_share": "{:.1%}"}))
print(f"Crude smoker-minus-non-smoker survival difference: {crude_difference_pp:+.2f} percentage points")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
y = list(range(len(baseline_summary)))
non_smoker_rates = baseline_summary["rate_non_smokers"]
smoker_rates = baseline_summary["rate_smokers"]
ax.hlines(y, smoker_rates, non_smoker_rates, color="#a9b4bd", linewidth=1.5)
ax.scatter(non_smoker_rates, y, marker="o", color="#356a8a", label="Non-smokers", zorder=3)
ax.scatter(smoker_rates, y, marker="D", color="#bf622d", label="Smokers", zorder=3)
for position, row in enumerate(baseline_summary.itertuples()):
    ax.annotate(
        f"{row.difference_pp:+.1f} pp",
        (max(row.rate_non_smokers, row.rate_smokers), position),
        xytext=(8, 0), textcoords="offset points", va="center", fontsize=9,
    )

ax.set_yticks(y, labels=baseline_summary.index)
ax.invert_yaxis()
ax.set_xlim(0, 1.15)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set_xlabel("Recorded survival proportion")
ax.set_title("Recorded survival by smoking status within age band")
ax.grid(axis="x", color="#e3e7eb")
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, loc="lower right")

with BytesIO() as image:
    fig.savefig(image, format="png", bbox_inches="tight")
    display(Image(
        data=image.getvalue(),
        alt="Paired survival proportions for smokers and non-smokers in four age bands. The smoker proportion is lower in each band, with the largest gap at ages 50 to 64.",
    ))
plt.close(fig)

## Interpretation

In the 1,314-record sample, the smoker-minus-non-smoker differences
were **−0.67 pp** at ages 18–29, **−4.14 pp** at 30–49, **−8.12 pp**
at 50–64, and **−2.51 pp** at 65+. The crude survival proportions
were **76.1%** for smokers and **68.6%** for non-smokers.

Smokers were younger in this sample. **8.6%** of smokers were 65+,
compared with **26.4%** of non-smokers. That difference in age
composition helps explain the crude comparison. Only **50** smokers
were in the 65+ band, including **6** recorded alive, so its rates
should be read with their denominators. The comparison is descriptive
and leaves other potential confounders unadjusted. See the
[sensitivity analysis](member_c_age_sensitivity.ipynb) for other
choices of age bands.